In [1]:
import os, sys, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

RUN_NAME = "v4-merged"

sec = UserSecretsClient()
for name in ("GIT_URL", "GITHUB_TOKEN", "GIT_BRANCH", "DAGSHUB_TOKEN"):
    try:
        os.environ[name] = sec.get_secret(name)
    except Exception:
        pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
BRANCH = os.environ.get("GIT_BRANCH", "main")
url = os.environ.get("GIT_URL", "")
token = os.environ.get("GITHUB_TOKEN", "")
if token and url.startswith("https://"):
    url = url.replace("https://", f"https://{token}@", 1)

if not (REPO / "run.py").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True))

!pip -q install transformers accelerate safetensors python-dotenv mlflow dagshub matplotlib

dagshub_token = os.environ.get("DAGSHUB_TOKEN", "").strip()
if not dagshub_token:
    raise SystemExit("Add Kaggle secret DAGSHUB_TOKEN")

env_lines = Path(".env.example").read_text().splitlines()
out_lines = []
for line in env_lines:
    if line.startswith("DAGSHUB_TOKEN="):
        out_lines.append(f"DAGSHUB_TOKEN={dagshub_token}")
    else:
        out_lines.append(line)
Path(".env").write_text("\n".join(out_lines) + "\n")
print("DagsHub on, run:", RUN_NAME)

Cloning into '/kaggle/working/filament-segmentation-2026'...


f0110e6 Merge remote-tracking branch 'origin/feature/eval-inference' into merge/v4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**GPU**

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Settings → GPU T4, Restart session"
x = torch.zeros(1, device="cuda")
print("ok", x.device)

True Tesla T4
ok cuda:0


**TRAIN**


In [3]:
%cd /kaggle/working/filament-segmentation-2026
!python run.py check
!python run.py train --size 1024 --epochs 8 --run-name v4-merged --eval-samples-val 100 --eval-samples-train 200
!ls -lh /kaggle/working/outputs/m2f/last.pt /kaggle/working/outputs/m2f/experiments/v4-merged/best.pt

/kaggle/working
check ok
preprocessor_config.json: 100%|████████████████| 537/537 [00:00<00:00, 2.39MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
config.json: 79.8kB [00:00, 42.9MB/s]
model.safetensors: 100%|█████████████████████| 190M/190M [00:12<00:00, 15.0MB/s]
Loading weights: 100%|█| 566/566 [00:00<00:00, 1504.05it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
class_predictor.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch

**METRICS**

In [4]:
from pathlib import Path
RUN = Path("/kaggle/working/outputs/m2f/experiments/v4-merged")
print((RUN / "metrics.csv").read_text())
print("good", len(list((RUN / "good").glob("*.png"))), "bad", len(list((RUN / "bad").glob("*.png"))))

epoch,train_loss,train_pq,val_pq
1,28.652734360763272,0.20702350282986823,0.2135465309855801
2,19.064958208542915,0.25495198456611384,0.2928278112756972
3,18.62465387924709,0.2267843379480243,0.25161546478399277
4,18.102341471082074,0.2665748486230258,0.2875306304008278
5,17.549177783642165,0.2860414104926177,0.2925018713969071
6,22.686553558439133,0.22023364446369947,0.24625408735570992
7,21.672051186045582,0.2382116069589405,0.26100627592619235
8,21.643571978597627,0.2384113522611142,0.25932200086865903

good 10 bad 7


**INFER + CSV**

In [5]:
%cd /kaggle/working/filament-segmentation-2026
!python run.py infer --submit --score-thr 0.3 --weights /kaggle/working/outputs/m2f/experiments/v4-merged/best.pt
!ls -lh /kaggle/working/outputs/m2f/submission_m2f.csv

/kaggle/working
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
Loading weights: 100%|█| 566/566 [00:00<00:00, 1445.96it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
class_predictor.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81, 256]) vs model:torch.Size([2, 256])
class_predictor.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81]) vs model:torch.Size([2])          
criterion.empty_weight | MISMATCH | Rein